# Домашнее задание 9

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

## Немного BackProp (1 балл)

Рассмотрим граф вычисления, который состоит из $N$ блоков, следующих друг за другом:

![alt](../data/Untitled%20Diagram.svg)

Допустим, что мы знаем производную лосс-функции $L$ по выходу из последнего блока $x_N$. Т.е. мы знаем $\frac{\partial L}{\partial x_N}$. 

**Задание:** Найдите производную $\frac{\partial L}{\partial x_0}$, если известны все производные $\frac{\partial f_i}{\partial x_i}$

Заппишите ответ в поле ниже в виде latex формулы:

## CNN-Разминка (1 балл)

Вспомним как работает сверточный слой:
1. на вход подается массив изображений, еще он называется батчем
2. к каждому изображению по границам добавляются нули
3. по каждому изображению "скользит" каждый из фильтров сверточного слоя

**Задание:** Необходимо реализовать функцию, добавляющую padding

Пример как это должно работать:

Пусть у нас есть батч `input_images` из двух изображений с тремя каналами (RGB). Размер изображений пусть будет 3*3. Вспомним, что вход сверточного слоя имеет следующую размерность:

* размер батча 
* число каналов
* высота
* ширина

В рассматриваемом случае размерность входа $(2, 3, 3, 3)$.

Если мы добавим вокруг каждого изображения отступ из одного нуля, то размер каждого изображений станет 3+2*1 = 5 пикселей в ширину и 5 в высоту соответственно (добавляем по одному нулю с каждой стороны изображения).

![alt](../data/pad.avif)

**Примечание:** Напишите любую работающую реализацию. Возможно, вам пригодится [документация](https://docs.pytorch.org/docs/stable/generated/torch.nn.ConstantPad2d.html#torch.nn.ConstantPad2d).

In [ ]:
import torch
import torch.nn as nn

def get_padding2d(input_images):
    padded_images = # нужно добавить нулей с четырех сторон каждого изображения
    return padded_images

In [ ]:
input_images = torch.tensor(
      [[[[0,  1,  2],
         [3,  4,  5],
         [6,  7,  8]],

        [[9, 10, 11],
         [12, 13, 14],
         [15, 16, 17]],

        [[18, 19, 20],
         [21, 22, 23],
         [24, 25, 26]]],


       [[[27, 28, 29],
         [30, 31, 32],
         [33, 34, 35]],

        [[36, 37, 38],
         [39, 40, 41],
         [42, 43, 44]],

        [[45, 46, 47],
         [48, 49, 50],
         [51, 52, 53]]]])

correct_padded_images = torch.tensor(
       [[[[0.,  0.,  0.,  0.,  0.],
          [0.,  0.,  1.,  2.,  0.],
          [0.,  3.,  4.,  5.,  0.],
          [0.,  6.,  7.,  8.,  0.],
          [0.,  0.,  0.,  0.,  0.]],

         [[0.,  0.,  0.,  0.,  0.],
          [0.,  9., 10., 11.,  0.],
          [0., 12., 13., 14.,  0.],
          [0., 15., 16., 17.,  0.],
          [0.,  0.,  0.,  0.,  0.]],

         [[0.,  0.,  0.,  0.,  0.],
          [0., 18., 19., 20.,  0.],
          [0., 21., 22., 23.,  0.],
          [0., 24., 25., 26.,  0.],
          [0.,  0.,  0.,  0.,  0.]]],


        [[[0.,  0.,  0.,  0.,  0.],
          [0., 27., 28., 29.,  0.],
          [0., 30., 31., 32.,  0.],
          [0., 33., 34., 35.,  0.],
          [0.,  0.,  0.,  0.,  0.]],

         [[0.,  0.,  0.,  0.,  0.],
          [0., 36., 37., 38.,  0.],
          [0., 39., 40., 41.,  0.],
          [0., 42., 43., 44.,  0.],
          [0.,  0.,  0.,  0.,  0.]],

         [[0.,  0.,  0.,  0.,  0.],
          [0., 45., 46., 47.,  0.],
          [0., 48., 49., 50.,  0.],
          [0., 51., 52., 53.,  0.],
          [0.,  0.,  0.,  0.,  0.]]]])

assert torch.allclose(get_padding2d(input_images), correct_padded_images)

## Реализация свёрточного слоя (1 балл)

Сейчас детально рассмотрим из чего состоит сверточный слой.

Сверточный слой это массив фильтров.

Каждый фильтр имеет следующую размерность:
* $C_{in}$ число каналов во входном изображении (для RGB это 3)
* $H$ высота фильтра
* $W$ ширина фильтра

В ядре (`kernel`) все фильтры имеют одинаковые размерность, поэтому ширину и высоту фильтров называют шириной и высотой ядра. Чаще всего ширина ядра равна высоте ядра, в таком случае их называют размером ядра (`kernel_size`).

Также слой имеет параметры:

* `padding` - на какое количество пикселей увеличивать входное изображение с каждой стороны.
* `stride` - на сколько пикселей смещается фильтр при вычислении свертки

**Задание:** Попробуйте самостоятельно вывести формулу размерности выхода сверточного слоя, зная параметры входа и ядра. Реализуйте функцию, принимающую на вход:
* Входную размерность (`input_matrix_shape`)
* Количество фильтров (`out_channels`)
* Размер одного фильтра (`kernel_size`)
* `padding`
* `stride`

**Примечание:** Правильность формулы проверьте, сравнив ее с формулой из [документации](https://pytorch.org/docs/stable/nn.html#torch.nn.Conv2d).

In [ ]:
import numpy as np
import torch
import torch.nn as nn


def calc_out_shape(input_matrix_shape, out_channels, kernel_size, stride, padding) -> list[int]:
    out_shape = # напишите тут код, вычисляющий выходную размерность

    return out_shape

In [ ]:
assert np.array_equal(
    calc_out_shape(input_matrix_shape=[2, 3, 10, 10],
                   out_channels=10,
                   kernel_size=3,
                   stride=1,
                   padding=0),
    [2, 10, 8, 8])

assert np.array_equal(
    calc_out_shape(input_matrix_shape=[2, 3, 10, 10],
                   out_channels=10,
                   kernel_size=3,
                   stride=1,
                   padding=1),
    [2, 10, 10, 10])

assert np.array_equal(
    calc_out_shape(input_matrix_shape=[2, 3, 10, 10],
                   out_channels=10,
                   kernel_size=3,
                   stride=2,
                   padding=0),
    [2, 10, 4, 4])

assert np.array_equal(
    calc_out_shape(input_matrix_shape=[2, 3, 10, 10],
                   out_channels=10,
                   kernel_size=1,
                   stride=2,
                   padding=0),
    [2, 10, 5, 5])

assert np.array_equal(
    calc_out_shape(input_matrix_shape=[2, 3, 10, 10],
                   out_channels=10,
                   kernel_size=10,
                   stride=2,
                   padding=0),
    [2, 10, 1, 1])

assert np.array_equal(
    calc_out_shape(input_matrix_shape=[2, 3, 10, 10],
                   out_channels=10,
                   kernel_size=2,
                   stride=1,
                   padding=0),
    [2, 10, 9, 9])

print("Good job")

## AlexNet (1 балл)

Сеть [AlexNet](https://proceedings.neurips.cc/paper_files/paper/2012/file/c399862d3b9d6b76c8436e924a68c45b-Paper.pdf) представляет собой первую широко известную глубокую (12 слоёв) свёрточную нейросеть, ставшую победителем соревнований ILSVRC в 2012 году с большим отрывом.

> Этот момент считается рождением глубокого обучения, поскольку победа AlexNet привлекла массовое внимание к глубоким нейросетям, и только глубокие нейросети стали побеждать в последующих соревнованиях ILSVRC.

Архитектура сети представляет собой усложненную версию LeNet. В реальности использовались ещё слои `Dropout`, но для нашей задачи подойдет и такая архитектура:

![alt](../data/AlexNetHome.png)

Настройка такой большой сети стала возможной за счёт использования видеокарт (graphics processing unit, GPU), позволивших производить векторные операции за один такт времени, используя параллельные вычисления на многих встроенных ядрах. Память видеокарт того времени (GTX 580 с 3 GB памя­ти) не позволила разместить нейросеть целиком, поэтому авторам пришлось использовать групповые свёртки, разбив число каналов пополам, причём каждая половина обрабатывалась на своей видеокарте. 


**Задание:** 
Ваша задача собрать AlexNet по картинке. На ней есть все размеры, которые должны иметь блоки в сверточной сети и с какими параметрами они задаются. Однако есть важные дополнения:

Дополнения:
1. В финальном слое должно быть не 1000 слоев, а всего 2
2. Финальный слой необходимо оставть `Linear(4096, 2)`
3. После каждого слоя `Conv2d` и `Linear` должна стоять активация `ReLU` (кроме финального).
4. При переходе от конволюций к линейным слоям нужно использовать слой `Flatten` (который дан в шаблоне), остальные слои берите из модуля `torch.nn`


In [ ]:
# a special module that converts [batch, channel, w, h] to [batch, units]
class Flatten(nn.Module):
    def forward(self, input):
        return input.view(input.size(0), -1)

def get_model():
    ReLU = nn.ReLU()
    return nn.Sequential(
        # Ваша реализация
    )

In [ ]:
model = get_model()

assert isinstance(model, nn.Sequential)
assert len(model) >= 15

first_layer = model[0]
assert isinstance(first_layer, nn.Conv2d)
assert first_layer.in_channels == 3
assert first_layer.out_channels == 96
assert first_layer.kernel_size == (11, 11)
assert first_layer.stride == (4, 4)

pool_layers = [i for i, layer in enumerate(model) if isinstance(layer, nn.MaxPool2d)]
assert len(pool_layers) >= 2

last_layer = model[-1]
assert isinstance(last_layer, nn.Linear)
assert last_layer.out_features == 2

assert any(isinstance(layer, Flatten) for layer in model)

In [ ]:
model = get_model()
model.eval()

# Создаём случайный батч: 4 изображения 227x227x3
batch_size = 4
x = torch.randn(batch_size, 3, 227, 227)

output = model(x)
assert isinstance(output, torch.Tensor)

expected_shape = (batch_size, 2)
assert output.shape == expected_shape
assert not torch.isnan(output).any()
assert not torch.isinf(output).any()

## Fashion-Mnist (1 балл)

Это задание творческое. Вам необходимо научиться реализовывать и обучать простейшую свёрточную нейронную сеть для задачи классификации. Делать вы это будете на изображениях одежды и аксессуаров из датасета `Fashion-MNIST`.

`Fashion-MNIST` — это набор из 70 000 чёрно-белых изображений размером 28×28 пикселей, разделённых на 10 классов:

0. T-shirt/top (футболка)
1. Trouser (брюки)
2. Pullover (пуловер)
3. Dress (платье)
4. Coat (пальто)
5. Sandal (сандалии)
6. Shirt (рубашка)
7. Sneaker (кроссовки)
8. Bag (сумка)
9. Ankle boot (ботинки)

Проведите несколько экспериментов. Попробуйте добиться лучшего качества. Можете использовать любую предобработку данных, тьюнить параметры, добавлять `Dropout`, регуляризацию, batch-norm и т.д. и т.п.

Необходимо написать краткий отчёт о проделанной вами работе. В отчете (т.е. в текстовой ячейке ниже вашего решения) надо кратко написать, что именно вы сделали, какие техники для улучешния результата вы успешно или неуспешно применили.

In [ ]:
# Task5
# Комментарий выше не стирать!
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

BATCH_SIZE = 64
EPOCHS = 5
LEARNING_RATE = 0.001

In [ ]:
# Task5
# Комментарий выше не стирать!
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor()  # только в тензор и /255
])

train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# Task5
# Комментарий выше не стирать!
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Ваша реализация

    def forward(self, x):
        # Ваша реализация
        return x

In [ ]:
# Task5
# Комментарий выше не стирать!
model = SimpleCNN().to(device)
print(model)

In [ ]:
# Task5
# Комментарий выше не стирать!

In [ ]:
# Task5
# Комментарий выше не стирать!

In [ ]:
# Task5
# Комментарий выше не стирать!

### Место для отчёта:


